# Pipeline Geografica — Geocodifica + Distanze per Power BI

## Struttura del notebook

| # | Blocco | Cosa produce |
|---|--------|--------------|
| 0 | Installazione dipendenze | — |
| 1 | Configurazione | Parametri globali |
| 2 | Funzioni di utilità | Helper riutilizzabili |
| 3 | Caricamento sedi esistenti | `df_esistenti` |
| 4 | Geocodifica sedi esistenti | `df_esistenti` con lat/lon |
| 5 | Caricamento nuove sedi | `df_nuove` |
| 6 | Geocodifica nuove sedi | `df_nuove` con lat/lon |
| 7 | Calcolo distanze (BallTree) | `df_distanze` |
| 8 | Export file per Power BI | 3 file Excel/CSV |
| 9 | Report riepilogativo | Statistiche finali |

---
**File di output generati:**
- `dim_sedi_esistenti.xlsx` — dimensione sedi esistenti con coordinate
- `dim_nuove_sedi.xlsx` — dimensione nuove sedi con coordinate
- `fact_distanze.csv` — tabella distanze (solo coppie entro soglia massima)


---
## BLOCCO 0 — Installazione dipendenze

Esegui questo blocco solo la prima volta, oppure se cambi ambiente/virtualenv.

In [ ]:
# ============================================================
# BLOCCO 0 — Installazione dipendenze
# ============================================================
# Esegui una volta sola. Se usi conda: sostituisci pip con conda.

import subprocess, sys

packages = [
    "pandas",
    "openpyxl",
    "geopy",
    "scikit-learn",   # BallTree per distanze geografiche veloci
    "numpy",
    "tqdm",           # barra avanzamento durante geocodifica
    "ipywidgets",     # necessario per tqdm.auto in Jupyter
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Tutte le dipendenze sono installate.")

---
## BLOCCO 1 — Configurazione

**Modifica solo questo blocco** per adattare il notebook al tuo ambiente.

In [ ]:
# ============================================================
# BLOCCO 1 — Configurazione centralizzata
# ============================================================
# Tutti i blocchi successivi leggono da qui.
# Non toccare nient'altro se non hai esigenze particolari.

from pathlib import Path

# ── File di INPUT ────────────────────────────────────────────
# Output di main.py già geocodificato
FILE_ESISTENTI   = Path("sedi_geocodificate.xlsx")

# Coordinate recuperate manualmente via Copilot
FILE_COPILOT     = Path("Punti fisici Copilot.xlsx")
SHEET_COPILOT    = "PUNTI FISICI"

# Nuove sedi da confrontare
# Cambia in "sedi_prova_2000.xlsx" per il volume grande
FILE_NUOVE       = Path("sedi_prova.xlsx")
SHEET_NUOVE      = "Sedi"

# ── File di OUTPUT ───────────────────────────────────────────
OUTPUT_DIR          = Path("output_powerbi")
FILE_OUT_ESISTENTI  = OUTPUT_DIR / "dim_sedi_esistenti.xlsx"
FILE_OUT_NUOVE      = OUTPUT_DIR / "dim_nuove_sedi.xlsx"
FILE_OUT_DISTANZE   = OUTPUT_DIR / "fact_distanze.csv"

# ── Cache geocodifica ────────────────────────────────────────
# JSON persistente: evita di rifare chiamate già fatte
FILE_CACHE_GEO      = Path("geocache.json")

# ── Parametri geocodifica Nominatim ──────────────────────────
USER_AGENT          = "pipeline-sedi-powerbi/2.0"
REQUEST_DELAY_SEC   = 1.1
MAX_RETRIES         = 3

# ── Parametri distanze ───────────────────────────────────────
# Coppie oltre questo raggio NON vengono salvate.
# In Power BI potrai usare lo slicer fino a questo valore massimo.
RAGGIO_MAX_KM       = 50.0

# ── Colonne attese nei file ──────────────────────────────────
COLONNE_ESISTENTI = [
    "Tipologia", "STATO PDV", "Rag. Sociale", "Codice Agenzia",
    "Codici FR", "Indirizzo", "Comune", "CAP", "Provincia",
    "Regione", "Sede Operativa SUP",
    "Latitudine", "Longitudine", "Stato_Geocoding",
]
COLONNE_NUOVE = [
    "Indirizzo", "Comune", "CAP", "Provincia", "Regione",
]
COLONNE_LAT_COPILOT = ["LAT", "Latitudine"]
COLONNE_LON_COPILOT = ["LON", "Longitudine"]

print("Configurazione caricata.")
print(f"  File esistenti : {FILE_ESISTENTI}")
print(f"  File Copilot   : {FILE_COPILOT}")
print(f"  File nuove     : {FILE_NUOVE}")
print(f"  Output dir     : {OUTPUT_DIR}")
print(f"  Raggio max     : {RAGGIO_MAX_KM} km")

---
## BLOCCO 2 — Funzioni di utilità

Raccoglie tutte le funzioni riutilizzabili: pulizia valori, normalizzazione
abbreviazioni stradali italiane, costruzione query geocodifica, cache
persistente JSON, geocodifica con retry, Haversine scalare.

Non produce output visibile. Deve essere eseguito prima degli altri blocchi.

In [ ]:
# ============================================================
# BLOCCO 2 — Funzioni di utilità
# ============================================================

import json
import logging
import time
import re

import numpy as np
import pandas as pd
from geopy.exc import GeocoderServiceError, GeocoderTimedOut, GeopyError
from geopy.geocoders import Nominatim

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ── 2a. Pulizia valori Excel ─────────────────────────────────

def clean_value(value) -> str:
    """
    Converte qualsiasi valore Excel (NaN, float con .0, testo)
    in stringa pulita, adatta per costruire query geocodifica.
    """
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


# ── 2b. Normalizzazione abbreviazioni stradali italiane ──────
# Espandere le abbreviazioni migliora il tasso di successo di Nominatim.

_ABBREV = {
    r"\bV\.le\b":  "Viale",
    r"\bC\.so\b":  "Corso",
    r"\bP\.za\b":  "Piazza",
    r"\bP\.le\b":  "Piazzale",
    r"\bF\.lli\b": "Fratelli",
    r"\bS\.\b":    "San",
    r"\bLg\.\b":   "Largo",
    r"\bLgo\b":    "Largo",
    r"\bV\.\b":    "Via",
}

def normalize_street(text: str) -> str:
    for pattern, replacement in _ABBREV.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


# ── 2c. Costruzione query geocodifica a tre livelli ──────────

def build_queries(row: pd.Series) -> tuple:
    """
    Ritorna tre query in ordine di precisione decrescente:
    1. completa  (via + CAP + comune + provincia + regione)
    2. fallback  (via + comune + provincia)
    3. comunale  (solo comune + provincia + regione)
    """
    via  = normalize_street(clean_value(row.get("Indirizzo", "")))
    cap  = clean_value(row.get("CAP", ""))
    com  = clean_value(row.get("Comune", ""))
    prov = clean_value(row.get("Provincia", ""))
    reg  = clean_value(row.get("Regione", ""))

    q_full     = f"{via}, {cap} {com}, {prov}, {reg}, Italia"
    q_fallback = f"{via}, {com}, {prov}, Italia"
    q_comune   = f"{com}, {prov}, {reg}, Italia"
    return q_full, q_fallback, q_comune


# ── 2d. Cache geocodifica persistente ────────────────────────
# Struttura JSON:
# { "query_string": {"lat": float, "lon": float,
#                    "address": str, "status": str, "precisione": str} }

def load_cache(path: Path) -> dict:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        log.info("Cache caricata: %d voci da %s", len(data), path)
        return data
    log.info("Cache non trovata, verrà creata: %s", path)
    return {}

def save_cache(cache: dict, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


# ── 2e. Chiamata Nominatim con retry esponenziale ────────────

def geocode_with_retry(geolocator: Nominatim, query: str) -> tuple:
    """
    Esegue la chiamata Nominatim con retry.
    Distingue errori transitori (timeout) da errori permanenti.
    Ritorna (location_object, error_string).
    """
    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            loc = geolocator.geocode(
                query, country_codes="it",
                addressdetails=True, timeout=10,
            )
            time.sleep(REQUEST_DELAY_SEC)
            return loc, ""
        except (GeocoderTimedOut, GeocoderServiceError) as exc:
            last_error = str(exc)
            log.warning("Tentativo %d/%d fallito per '%s': %s",
                        attempt, MAX_RETRIES, query, last_error)
            time.sleep(REQUEST_DELAY_SEC * attempt)
        except GeopyError as exc:
            last_error = str(exc)
            log.error("Errore permanente per '%s': %s", query, last_error)
            break
    return None, last_error


# ── 2f. Geocodifica singola riga con cache ───────────────────

def geocode_row(geolocator: Nominatim, row: pd.Series, cache: dict) -> dict:
    """
    Geocodifica una riga del DataFrame.
    1. Controlla la cache (evita chiamate ripetute).
    2. Prova le tre query in cascata (completa → fallback → comunale).
    3. Salva ogni risultato trovato nella cache.

    Ritorna dict con chiavi:
    Latitudine, Longitudine, Indirizzo_Trovato,
    Stato_Geocoding, Precisione_Geocoding
    """
    q_full, q_fb, q_com = build_queries(row)

    if not clean_value(row.get("Indirizzo", "")) or not clean_value(row.get("Comune", "")):
        return {
            "Latitudine": None, "Longitudine": None,
            "Indirizzo_Trovato": "",
            "Stato_Geocoding": "DATI_INSUFFICIENTI",
            "Precisione_Geocoding": "NESSUNA",
        }

    levels = [
        (q_full, "TROVATO",          "CIVICO"),
        (q_fb,   "TROVATO_FALLBACK", "VIA"),
        (q_com,  "TROVATO_COMUNE",   "COMUNE"),
    ]

    # Controlla prima la cache
    for query, stato, precisione in levels:
        if query in cache:
            hit = cache[query]
            return {
                "Latitudine":            hit["lat"],
                "Longitudine":           hit["lon"],
                "Indirizzo_Trovato":    hit["address"],
                "Stato_Geocoding":      hit.get("status", stato),
                "Precisione_Geocoding": hit.get("precisione", precisione),
            }

    # Non in cache: chiama Nominatim
    errors = []
    for query, stato, precisione in levels:
        loc, err = geocode_with_retry(geolocator, query)
        if err:
            errors.append(err)
        if loc:
            result = {
                "Latitudine":            loc.latitude,
                "Longitudine":           loc.longitude,
                "Indirizzo_Trovato":    loc.address,
                "Stato_Geocoding":      stato,
                "Precisione_Geocoding": precisione,
            }
            cache[query] = {
                "lat": loc.latitude, "lon": loc.longitude,
                "address": loc.address, "status": stato,
                "precisione": precisione,
            }
            save_cache(cache, FILE_CACHE_GEO)
            return result

    return {
        "Latitudine": None, "Longitudine": None,
        "Indirizzo_Trovato": "",
        "Stato_Geocoding": f"ERRORE: {errors[-1]}" if errors else "NON_TROVATO",
        "Precisione_Geocoding": "NESSUNA",
    }


# ── 2g. Normalizzazione colonne DataFrame ────────────────────

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.astype(str).str.strip()
    df = df.dropna(axis=1, how="all")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False, na=False)]
    return df


# ── 2h. Haversine scalare (per verifica/debug) ───────────────

def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


print("Funzioni di utilità definite.")

---
## BLOCCO 3 — Caricamento sedi esistenti

Legge il file già geocodificato da `main.py`, integra le coordinate
mancanti dal file Copilot (join per indirizzo normalizzato) e
costruisce `df_esistenti` pronto per il calcolo distanze.

In [ ]:
# ============================================================
# BLOCCO 3 — Caricamento e integrazione sedi esistenti
# ============================================================

if not FILE_ESISTENTI.exists():
    raise FileNotFoundError(
        f"File non trovato: {FILE_ESISTENTI}\n"
        "Assicurati di aver eseguito prima main.py."
    )

# ── 3a. Lettura file principale ──────────────────────────────
df_main = pd.read_excel(FILE_ESISTENTI, dtype=str)
df_main = normalize_df(df_main)
df_main["Latitudine"]  = pd.to_numeric(df_main.get("Latitudine"),  errors="coerce")
df_main["Longitudine"] = pd.to_numeric(df_main.get("Longitudine"), errors="coerce")

print(f"Sedi caricate dal file principale : {len(df_main):>4}")
print(f"Con coordinate                    : {df_main['Latitudine'].notna().sum():>4}")
print(f"Senza coordinate                  : {df_main['Latitudine'].isna().sum():>4}")

# ── 3b. Lettura file Copilot (opzionale) ─────────────────────
df_cop = None
if FILE_COPILOT.exists():
    df_cop_raw = pd.read_excel(FILE_COPILOT, sheet_name=SHEET_COPILOT, dtype=str)
    df_cop_raw = normalize_df(df_cop_raw)
    col_lat = next((c for c in COLONNE_LAT_COPILOT if c in df_cop_raw.columns), None)
    col_lon = next((c for c in COLONNE_LON_COPILOT if c in df_cop_raw.columns), None)

    if col_lat and col_lon and "Indirizzo" in df_cop_raw.columns:
        df_cop = df_cop_raw[["Indirizzo", col_lat, col_lon]].copy()
        df_cop.rename(columns={col_lat: "Lat_Cop", col_lon: "Lon_Cop"}, inplace=True)
        df_cop["Lat_Cop"] = pd.to_numeric(df_cop["Lat_Cop"], errors="coerce")
        df_cop["Lon_Cop"] = pd.to_numeric(df_cop["Lon_Cop"], errors="coerce")
        df_cop = (
            df_cop
            .dropna(subset=["Lat_Cop", "Lon_Cop"])
            .drop_duplicates(subset=["Indirizzo"], keep="first")
        )
        print(f"\nCoordinate Copilot disponibili : {len(df_cop)}")
    else:
        print("File Copilot trovato ma colonne non identificate.")
else:
    print("File Copilot non trovato — si procede solo con Nominatim.")

# ── 3c. Integrazione coordinate Copilot ─────────────────────
# Riempie solo le righe senza coordinate con quelle del file Copilot
if df_cop is not None:
    df_main = df_main.merge(df_cop, on="Indirizzo", how="left")
    mask_mancanti = df_main["Latitudine"].isna()
    df_main.loc[mask_mancanti, "Latitudine"]  = df_main.loc[mask_mancanti, "Lat_Cop"]
    df_main.loc[mask_mancanti, "Longitudine"] = df_main.loc[mask_mancanti, "Lon_Cop"]
    df_main["OrigineCoordinate"] = "Nominatim"
    df_main.loc[mask_mancanti & df_main["Latitudine"].notna(), "OrigineCoordinate"] = "Copilot"
    df_main.drop(columns=["Lat_Cop", "Lon_Cop"], errors="ignore", inplace=True)
    recuperate = (df_main["OrigineCoordinate"] == "Copilot").sum()
    print(f"Sedi recuperate da Copilot        : {recuperate:>4}")
else:
    df_main["OrigineCoordinate"] = "Nominatim"

# ── 3d. ID univoco e tipo punto ──────────────────────────────
df_main.insert(0, "SedeID", range(1, len(df_main) + 1))
df_main["TipoPunto"] = "Esistente"
df_esistenti = df_main.copy()

print(f"\nTOTALE sedi esistenti             : {len(df_esistenti):>4}")
print(f"Con coordinate valide             : {df_esistenti['Latitudine'].notna().sum():>4}")
print(f"Ancora senza coordinate           : {df_esistenti['Latitudine'].isna().sum():>4}")
df_esistenti[["SedeID","Indirizzo","Comune","Latitudine","Longitudine","OrigineCoordinate"]].head(5)

---
## BLOCCO 4 — Geocodifica sedi esistenti (opzionale)

Esegui solo se ci sono sedi esistenti ancora senza coordinate dopo il Blocco 3.
Se tutte le sedi hanno già lat/lon, questo blocco stampa un messaggio e termina.

In [ ]:
# ============================================================
# BLOCCO 4 — Geocodifica sedi esistenti mancanti
# ============================================================

from tqdm.auto import tqdm   # auto: funziona sia in Jupyter che in terminale

mask_mancanti = df_esistenti["Latitudine"].isna()
n_mancanti = mask_mancanti.sum()

if n_mancanti == 0:
    print("Tutte le sedi esistenti hanno già le coordinate. Blocco saltato.")
else:
    print(f"Sedi da geocodificare  : {n_mancanti}")
    print(f"Tempo stimato          : ~{n_mancanti * REQUEST_DELAY_SEC / 60:.1f} minuti")

    cache_geo  = load_cache(FILE_CACHE_GEO)
    geolocator = Nominatim(user_agent=USER_AGENT)

    for idx in tqdm(df_esistenti[mask_mancanti].index, desc="Geocodifica esistenti"):
        res = geocode_row(geolocator, df_esistenti.loc[idx], cache_geo)
        df_esistenti.loc[idx, "Latitudine"]           = res["Latitudine"]
        df_esistenti.loc[idx, "Longitudine"]          = res["Longitudine"]
        df_esistenti.loc[idx, "Indirizzo_Trovato"]    = res.get("Indirizzo_Trovato", "")
        df_esistenti.loc[idx, "Stato_Geocoding"]      = res["Stato_Geocoding"]
        df_esistenti.loc[idx, "Precisione_Geocoding"] = res["Precisione_Geocoding"]
        if res["Latitudine"] is not None:
            df_esistenti.loc[idx, "OrigineCoordinate"] = "Nominatim_NB"

    trovate  = df_esistenti["Latitudine"].notna().sum()
    print(f"\nCon coordinate dopo geocodifica : {trovate}")
    print(f"Ancora senza coordinate         : {df_esistenti['Latitudine'].isna().sum()}")

---
## BLOCCO 5 — Caricamento nuove sedi

Legge il file delle nuove sedi. Se contiene già colonne lat/lon le usa
direttamente; altrimenti le inizializza a `None` per la geocodifica.

In [ ]:
# ============================================================
# BLOCCO 5 — Caricamento nuove sedi
# ============================================================

if not FILE_NUOVE.exists():
    raise FileNotFoundError(f"File non trovato: {FILE_NUOVE}")

try:
    df_nuove_raw = pd.read_excel(FILE_NUOVE, sheet_name=SHEET_NUOVE, dtype=str)
except Exception:
    df_nuove_raw = pd.read_excel(FILE_NUOVE, sheet_name=0, dtype=str)
    print(f"Foglio '{SHEET_NUOVE}' non trovato, usato il primo foglio.")

df_nuove_raw = normalize_df(df_nuove_raw)

# Verifica colonne minime
col_mancanti = [c for c in COLONNE_NUOVE if c not in df_nuove_raw.columns]
if col_mancanti:
    print(f"Colonne mancanti: {col_mancanti}")
    print(f"Colonne disponibili: {list(df_nuove_raw.columns)}")

# Gestione coordinate già presenti (vari nomi possibili)
_lat_aliases = ["Latitudine", "LAT", "lat", "latitude"]
_lon_aliases = ["Longitudine", "LON", "lon", "longitude"]
col_lat_n = next((c for c in _lat_aliases if c in df_nuove_raw.columns), None)
col_lon_n = next((c for c in _lon_aliases if c in df_nuove_raw.columns), None)

if col_lat_n and col_lon_n:
    df_nuove_raw["Latitudine"]  = pd.to_numeric(df_nuove_raw[col_lat_n],  errors="coerce")
    df_nuove_raw["Longitudine"] = pd.to_numeric(df_nuove_raw[col_lon_n], errors="coerce")
    if col_lat_n != "Latitudine":
        df_nuove_raw.drop(columns=[col_lat_n, col_lon_n], errors="ignore", inplace=True)
    print(f"Coordinate trovate nel file ({col_lat_n}, {col_lon_n}).")
else:
    df_nuove_raw["Latitudine"]  = None
    df_nuove_raw["Longitudine"] = None
    print("Coordinate non presenti. Verranno geocodificate nel Blocco 6.")

df_nuove_raw.insert(0, "NuovaSedeID", range(1, len(df_nuove_raw) + 1))
df_nuove_raw["TipoPunto"]         = "Nuovo"
df_nuove_raw["OrigineCoordinate"] = None
df_nuove = df_nuove_raw.copy()

print(f"\nNuove sedi caricate  : {len(df_nuove):>5}")
print(f"Con coordinate       : {df_nuove['Latitudine'].notna().sum():>5}")
print(f"Da geocodificare     : {df_nuove['Latitudine'].isna().sum():>5}")
df_nuove[["NuovaSedeID","Indirizzo","Comune","Latitudine","Longitudine"]].head(5)

---
## BLOCCO 6 — Geocodifica nuove sedi

Geocodifica le nuove sedi senza coordinate. Usa la cache persistente.

> **Attenzione per file con 500+ sedi senza coordinate:**
> Nominatim non è adatto a volumi massivi (policy OSM).
> In quel caso usa `main_google.py` che hai già nel progetto,
> oppure fornisci il file con lat/lon già precompilate.

In [ ]:
# ============================================================
# BLOCCO 6 — Geocodifica nuove sedi
# ============================================================

from tqdm.auto import tqdm   # auto: funziona sia in Jupyter che in terminale

mask_nuove_mancanti = df_nuove["Latitudine"].isna()
n_da_geo = mask_nuove_mancanti.sum()

if n_da_geo == 0:
    print("Tutte le nuove sedi hanno già le coordinate. Blocco saltato.")
    for col in ["Stato_Geocoding", "Precisione_Geocoding"]:
        if col not in df_nuove.columns:
            df_nuove[col] = "PRECOMPILATO"
    df_nuove["OrigineCoordinate"] = "Precompilato"
else:
    if n_da_geo > 500:
        print(
            f"ATTENZIONE: {n_da_geo} sedi da geocodificare con Nominatim.\n"
            f"Tempo stimato: ~{n_da_geo * REQUEST_DELAY_SEC / 60:.0f} minuti.\n"
            f"Per volumi grandi usa main_google.py o fornisci lat/lon precompilate."
        )
    else:
        print(f"Nuove sedi da geocodificare : {n_da_geo}")
        print(f"Tempo stimato              : ~{n_da_geo * REQUEST_DELAY_SEC / 60:.1f} minuti")

    cache_geo  = load_cache(FILE_CACHE_GEO)
    geolocator = Nominatim(user_agent=USER_AGENT)

    for col in ["Stato_Geocoding", "Precisione_Geocoding", "Indirizzo_Trovato"]:
        if col not in df_nuove.columns:
            df_nuove[col] = None

    for idx in tqdm(df_nuove[mask_nuove_mancanti].index, desc="Geocodifica nuove sedi"):
        res = geocode_row(geolocator, df_nuove.loc[idx], cache_geo)
        df_nuove.loc[idx, "Latitudine"]           = res["Latitudine"]
        df_nuove.loc[idx, "Longitudine"]          = res["Longitudine"]
        df_nuove.loc[idx, "Indirizzo_Trovato"]    = res.get("Indirizzo_Trovato", "")
        df_nuove.loc[idx, "Stato_Geocoding"]      = res["Stato_Geocoding"]
        df_nuove.loc[idx, "Precisione_Geocoding"] = res["Precisione_Geocoding"]
        if res["Latitudine"] is not None:
            df_nuove.loc[idx, "OrigineCoordinate"] = "Nominatim"

    print(f"\nNuove sedi con coordinate   : {df_nuove['Latitudine'].notna().sum()}")
    print(f"Nuove sedi senza coordinate : {df_nuove['Latitudine'].isna().sum()}")

---
## BLOCCO 7 — Calcolo distanze con BallTree

**Cuore della pipeline.** Usa `BallTree` con metrica Haversine per trovare,
per ogni nuova sede, tutte le sedi esistenti entro `RAGGIO_MAX_KM`.

**Perché BallTree invece del cross join di Power Query?**

| Metodo | 2.000 × 275 | 20.000 × 275 | Complessità |
|--------|-------------|--------------|-------------|
| Cross join Power Query | ~5 min | ~60 min+ | O(n×m) |
| BallTree Python | < 1 sec | < 5 sec | O(n log m) |

In più BallTree produce **solo le coppie entro il raggio**: zero righe
inutili, file più piccolo, Power BI più veloce.

In [ ]:
# ============================================================
# BLOCCO 7 — Calcolo distanze con BallTree
# ============================================================

from sklearn.neighbors import BallTree

RAGGIO_TERRA_KM = 6371.0

# ── 7a. Filtra solo punti con coordinate valide ──────────────
esi_valide   = df_esistenti.dropna(subset=["Latitudine", "Longitudine"]).copy()
nuove_valide = df_nuove.dropna(subset=["Latitudine", "Longitudine"]).copy()

print(f"Sedi esistenti con coordinate : {len(esi_valide):>5}")
print(f"Nuove sedi con coordinate     : {len(nuove_valide):>5}")
print(f"Raggio massimo                : {RAGGIO_MAX_KM} km")
print(f"Confronti massimi teorici     : {len(nuove_valide) * len(esi_valide):>8,}")

if len(nuove_valide) == 0 or len(esi_valide) == 0:
    raise ValueError(
        "Nessun punto valido per il calcolo distanze. "
        "Controlla la geocodifica nei blocchi precedenti."
    )

# ── 7b. Costruzione BallTree ─────────────────────────────────
# BallTree lavora in RADIANTI: converte lat/lon da gradi
coords_esi   = np.radians(esi_valide[["Latitudine", "Longitudine"]].values.astype(float))
coords_nuove = np.radians(nuove_valide[["Latitudine", "Longitudine"]].values.astype(float))

# Costruisce l'albero sulle sedi esistenti (operazione rapida, fatta una volta)
tree = BallTree(coords_esi, metric="haversine")
print(f"\nBallTree costruito su {len(esi_valide)} sedi esistenti.")

# ── 7c. Query: sedi esistenti entro raggio per ogni nuova sede
# Il raggio si converte da km a radianti: km / R_terra
raggio_rad = RAGGIO_MAX_KM / RAGGIO_TERRA_KM

# indices  = lista di array con gli indici posizionali delle sedi vicine
# distances = lista di array con le distanze in radianti (ordinate per crescente)
indices, distances = tree.query_radius(
    coords_nuove,
    r=raggio_rad,
    return_distance=True,
    sort_results=True,
)

# ── 7d. Costruzione tabella distanze ─────────────────────────
# Espande le coppie (nuova sede, sede esistente, distanza_km)
righe = []

for pos_nuova, (idx_list, dist_list) in enumerate(zip(indices, distances)):
    if len(idx_list) == 0:
        continue

    nuova_row = nuove_valide.iloc[pos_nuova]

    for pos_esi, dist_rad in zip(idx_list, dist_list):
        dist_km  = dist_rad * RAGGIO_TERRA_KM
        esi_row  = esi_valide.iloc[pos_esi]

        righe.append({
            # Chiavi di relazione per Power BI
            "NuovaSedeID":       int(nuova_row["NuovaSedeID"]),
            "SedeID":            int(esi_row["SedeID"]),
            # Distanza arrotondata a 3 decimali
            "DistanzaKm":        round(dist_km, 3),
            # Campi nuova sede (per tooltip Power BI)
            "Nuova_Indirizzo":   nuova_row.get("Indirizzo", ""),
            "Nuova_Comune":      nuova_row.get("Comune", ""),
            "Nuova_Provincia":   nuova_row.get("Provincia", ""),
            # Campi sede esistente (per tooltip Power BI)
            "Esi_CodiciFR":      esi_row.get("Codici FR", ""),
            "Esi_CodiceAgenzia": esi_row.get("Codice Agenzia", ""),
            "Esi_Indirizzo":     esi_row.get("Indirizzo", ""),
            "Esi_Comune":        esi_row.get("Comune", ""),
            "Esi_Provincia":     esi_row.get("Provincia", ""),
            "Esi_Tipologia":     esi_row.get("Tipologia", ""),
            "Esi_StatoPDV":      esi_row.get("STATO PDV", ""),
        })

df_distanze = pd.DataFrame(righe)

# ── 7e. Statistiche ──────────────────────────────────────────
nuove_con_vicini = df_distanze["NuovaSedeID"].nunique() if len(df_distanze) > 0 else 0

print(f"\nRighe nella Fact_Distanze     : {len(df_distanze):>8,}")
print(f"Nuove sedi con almeno 1 vicino: {nuove_con_vicini:>8,}")
print(f"Nuove sedi senza vicini       : {len(nuove_valide) - nuove_con_vicini:>8,}")
if len(df_distanze) > 0:
    print(f"Distanza media (km)           : {df_distanze['DistanzaKm'].mean():>8.2f}")
    print(f"Distanza minima trovata (km)  : {df_distanze['DistanzaKm'].min():>8.3f}")
df_distanze.head(5)

---
## BLOCCO 8 — Export file per Power BI

Genera i tre file nella cartella `output_powerbi/`.

| File | Tabella Power BI | Utilizzo |
|------|-----------------|----------|
| `dim_sedi_esistenti.xlsx` | `Dim_Sedi_Esistenti` | Mappa + filtri |
| `dim_nuove_sedi.xlsx` | `Dim_Nuove_Sedi` | Mappa + filtri |
| `fact_distanze.csv` | `Fact_Distanze` | Calcoli + slicer soglia |

In [ ]:
# ============================================================
# BLOCCO 8 — Export file per Power BI
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 8a. Dim_Sedi_Esistenti ───────────────────────────────────
colonne_dim_esi = [
    "SedeID", "TipoPunto",
    "Tipologia", "STATO PDV", "Rag. Sociale",
    "Codice Agenzia", "Codici FR",
    "Indirizzo", "Comune", "CAP", "Provincia", "Regione",
    "Sede Operativa SUP",
    "Latitudine", "Longitudine",
    "Stato_Geocoding", "Precisione_Geocoding", "OrigineCoordinate",
]
cols_ok = [c for c in colonne_dim_esi if c in df_esistenti.columns]
df_out_esi = df_esistenti[cols_ok].copy()
df_out_esi.to_excel(FILE_OUT_ESISTENTI, index=False, engine="openpyxl")
print(f"Dim_Sedi_Esistenti  -> {FILE_OUT_ESISTENTI}  ({len(df_out_esi)} righe)")

# ── 8b. Dim_Nuove_Sedi ───────────────────────────────────────
colonne_dim_nuove = [
    "NuovaSedeID", "TipoPunto",
    "Tipologia", "Rag. Sociale", "Codice Agenzia", "Codici FR",
    "Indirizzo", "Comune", "CAP", "Provincia", "Regione",
    "Sede Operativa SUP",
    "Latitudine", "Longitudine",
    "Stato_Geocoding", "Precisione_Geocoding", "OrigineCoordinate",
]
cols_ok_n = [c for c in colonne_dim_nuove if c in df_nuove.columns]
df_out_nuove = df_nuove[cols_ok_n].copy()
df_out_nuove.to_excel(FILE_OUT_NUOVE, index=False, engine="openpyxl")
print(f"Dim_Nuove_Sedi      -> {FILE_OUT_NUOVE}  ({len(df_out_nuove)} righe)")

# ── 8c. Fact_Distanze (CSV per file potenzialmente grandi) ───
if len(df_distanze) > 0:
    df_distanze.to_csv(FILE_OUT_DISTANZE, index=False, encoding="utf-8-sig")
    print(f"Fact_Distanze       -> {FILE_OUT_DISTANZE}  ({len(df_distanze):,} righe)")
else:
    print(f"Fact_Distanze vuota: nessuna coppia entro {RAGGIO_MAX_KM} km.")

# ── 8d. Riepilogo ────────────────────────────────────────────
print(f"\nCartella output: {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size_kb:>7.1f} KB")

---
## BLOCCO 9 — Report riepilogativo

Statistiche finali: qualità geocodifica, distribuzione distanze
per soglie comuni, top sedi esistenti più frequentate.

In [ ]:
# ============================================================
# BLOCCO 9 — Report riepilogativo
# ============================================================

print("=" * 55)
print("REPORT FINALE PIPELINE GEOCODIFICA + DISTANZE")
print("=" * 55)

print("\nSEDI ESISTENTI")
print(f"  Totale                   : {len(df_esistenti):>5}")
print(f"  Con coordinate           : {df_esistenti['Latitudine'].notna().sum():>5}")
print(f"  Senza coordinate         : {df_esistenti['Latitudine'].isna().sum():>5}")

if "Precisione_Geocoding" in df_esistenti.columns:
    print("  Precisione:")
    for val, cnt in df_esistenti["Precisione_Geocoding"].value_counts().items():
        print(f"    {str(val):<20}: {cnt:>4}")

print("\nNUOVE SEDI")
print(f"  Totale                   : {len(df_nuove):>5}")
print(f"  Con coordinate           : {df_nuove['Latitudine'].notna().sum():>5}")
print(f"  Senza coordinate         : {df_nuove['Latitudine'].isna().sum():>5}")

if "Precisione_Geocoding" in df_nuove.columns:
    print("  Precisione:")
    for val, cnt in df_nuove["Precisione_Geocoding"].value_counts().items():
        print(f"    {str(val):<20}: {cnt:>4}")

if len(df_distanze) > 0:
    print(f"\nDISTANZE (raggio max: {RAGGIO_MAX_KM} km)")
    print(f"  Coppie totali            : {len(df_distanze):>8,}")

    print("\n  Nuove sedi con almeno 1 sede esistente entro:")
    for soglia in [1, 2, 5, 10, 20, 50]:
        if soglia > RAGGIO_MAX_KM:
            break
        n = df_distanze[df_distanze["DistanzaKm"] <= soglia]["NuovaSedeID"].nunique()
        pct = n / len(nuove_valide) * 100 if len(nuove_valide) > 0 else 0
        print(f"    {str(soglia) + ' km':<8}: {n:>5}  ({pct:.1f}%)")

    print("\n  Top 5 sedi esistenti con piu' nuove sedi vicine (entro 5 km):")
    top_esi = (
        df_distanze[df_distanze["DistanzaKm"] <= 5]
        .groupby(["SedeID", "Esi_CodiciFR", "Esi_Comune"])
        .agg(NumNuoveVicine=("NuovaSedeID", "nunique"))
        .sort_values("NumNuoveVicine", ascending=False)
        .head(5)
        .reset_index()
    )
    print(top_esi.to_string(index=False))
else:
    print("\nNessuna coppia trovata entro il raggio massimo.")

print("\n" + "=" * 55)
print("Pipeline completata. File pronti in:", OUTPUT_DIR.resolve())
print("=" * 55)

---
## Istruzioni per Power BI

### 1. Importa i 3 file
- **Home → Recupera dati → Excel**: importa `dim_sedi_esistenti.xlsx` e `dim_nuove_sedi.xlsx`
- **Home → Recupera dati → Testo/CSV**: importa `fact_distanze.csv`

### 2. Crea il parametro soglia (What-If)
**Creazione modello → Nuovo parametro → Intervallo numerico**
- Nome: `Soglia_Km` | Min: 1 | Max: 50 | Incremento: 1 | Default: 5
- Power BI crea automaticamente la tabella `Soglia_Km` con lo slicer

### 3. Relazioni nel modello dati
```
Fact_Distanze[NuovaSedeID]  →  Dim_Nuove_Sedi[NuovaSedeID]   (molti a uno)
Fact_Distanze[SedeID]       →  Dim_Sedi_Esistenti[SedeID]     (molti a uno)
```

### 4. Misure DAX da creare

```dax
-- Sedi esistenti vicine entro la soglia selezionata
Sedi Vicine Entro Soglia =
CALCULATE(
    COUNTROWS(Fact_Distanze),
    Fact_Distanze[DistanzaKm] <= [Soglia_Km Valore]
)

-- Distanza minima per la nuova sede selezionata
Distanza Minima Km =
CALCULATE(
    MIN(Fact_Distanze[DistanzaKm]),
    ALLEXCEPT(Fact_Distanze, Fact_Distanze[NuovaSedeID])
)

-- Flag per colorare i punti nella mappa
Flag Entro Soglia =
IF([Sedi Vicine Entro Soglia] > 0, "Entro soglia", "Fuori soglia")
```

### 5. Visual consigliato
- Usa **Azure Maps** in Power BI
- Layer 1 (punti blu): `Dim_Sedi_Esistenti` — Latitudine + Longitudine
- Layer 2 (punti arancio): `Dim_Nuove_Sedi` — Latitudine + Longitudine
- Colore per `[Flag Entro Soglia]` sulle nuove sedi
- Tooltip: Indirizzo, Comune, `[Distanza Minima Km]`, `[Sedi Vicine Entro Soglia]`
- Slicer: parametro `Soglia_Km`
